### Tools

In [2]:
import os
from langchain.tools import tool
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [16]:
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("What's the weather like in Chennai today?")
print(response.content)

<think>
Okay, the user is asking about the weather in Chennai today. Let me think about how to approach this.

First, I know that I can't access real-time data, so I can't provide current weather updates. I should inform them about that. But maybe I can suggest how they can check the weather. 

They might not know about weather apps or websites, so listing some options would be helpful. Maybe mention popular ones like Weather.com, AccuWeather, or even Google Weather. Also, remind them to enable location services for accurate results. 

Alternatively, they could use voice assistants like Siri or Alexa. Maybe include that as an option too. 

I should also consider their location. Chennai is in India, so maybe mention local services like the India Meteorological Department (IMD) website. That could be more reliable for local forecasts. 

Wait, but the user might prefer an app they already have. Maybe list a few different methods so they can choose what's convenient. 

I need to make sure 

In [17]:
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

In [20]:
model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather like in Chennai today?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']} with arguments {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Chennai today. Let me check what tools I have. There\'s a function called get_weather that takes a location parameter. Since the user specified Chennai, I need to call that function with "Chennai" as the location. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': '37057yf9e', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 157, 'total_tokens': 257, 'completion_time': 0.14972501, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.006406894, 'prompt_tokens_details': None, 'queue_time': 0.047940463, 'total_time': 0.156131904}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logp

### Tool Execution Loops

In [22]:
messages = [{"role":"user", "content": "What's the weather like in Chennai today?"}]
print(f"User message: {messages[-1]['content']}")
ai_msg = model_with_tools.invoke(messages)
print(f"AI response: {ai_msg}")
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    print(f"Tool result: {tool_result}")
    messages.append(tool_result)

print(f"Post tool messages: {messages}")

final_response = model_with_tools.invoke(messages)
print(f"Final AI response: {final_response.text}")

User message: What's the weather like in Chennai today?
AI response: content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Chennai today. I need to use the get_weather function. The function requires a location parameter. Chennai is the location here. So I should call get_weather with location set to "Chennai". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'ba4pq9faq', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 157, 'total_tokens': 254, 'completion_time': 0.163080332, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.016282366, 'prompt_tokens_details': None, 'queue_time': 0.04810088, 'total_time': 0.179362698}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'f

In [24]:
messages

[{'role': 'user', 'content': "What's the weather like in Chennai today?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Chennai today. I need to use the get_weather function. The function requires a location parameter. Chennai is the location here. So I should call get_weather with location set to "Chennai". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'ba4pq9faq', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 157, 'total_tokens': 254, 'completion_time': 0.163080332, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.016282366, 'prompt_tokens_details': None, 'queue_time': 0.04810088, 'total_time': 0.179362698}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tie